# 05 - Historical Walk-Forward Evaluation

This notebook evaluates the return series generated by the rolling allocation. The exercise is historical walk-forward rather than a genuinely untouched out-of-sample test.

## 1. Load results

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
current_directory = Path.cwd()
repository_root = current_directory.parent if current_directory.name == "notebooks" else current_directory

if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from src import active_metrics, annualized_return, load_config, performance_metrics

In [ ]:
config = load_config(repository_root / "config" / "config.yaml")
tables_directory = repository_root / "results" / "tables"
figures_directory = repository_root / "results" / "figures"

walk_forward_returns = pd.read_csv(
    tables_directory / "walk_forward_returns.csv",
    index_col="date",
    parse_dates=True
)
walk_forward_gross_returns = pd.read_csv(
    tables_directory / "walk_forward_gross_returns.csv",
    index_col="date",
    parse_dates=True
)
walk_forward_turnover = pd.read_csv(
    tables_directory / "walk_forward_turnover.csv",
    index_col="date",
    parse_dates=True
)

## 2. Performance and active results

In [ ]:
methods = [column for column in walk_forward_returns if column != config["benchmark"]]
benchmark_returns = walk_forward_returns[config["benchmark"]]

performance_summary = walk_forward_returns.apply(performance_metrics).T
active_summary = pd.DataFrame({
    method: active_metrics(walk_forward_returns[method], benchmark_returns)
    for method in methods
}).T
performance_summary = performance_summary.join(active_summary)

gross_annualized_returns = walk_forward_gross_returns[methods].apply(annualized_return)
performance_summary.loc[methods, "annualized_cost_drag"] = (
    gross_annualized_returns - performance_summary.loc[methods, "annualized_return"]
)
performance_summary.loc[methods, "annualized_turnover"] = walk_forward_turnover[methods].mean() * 12

performance_summary

In [ ]:
live_start = pd.Timestamp("2014-01-31")
live_returns = walk_forward_returns.loc[live_start:]
live_performance_summary = live_returns.apply(performance_metrics).T
live_active_summary = pd.DataFrame({
    method: active_metrics(live_returns[method], live_returns[config["benchmark"]])
    for method in methods
}).T
live_performance_summary = live_performance_summary.join(live_active_summary)

live_performance_summary

## 3. Block-bootstrap uncertainty

In [ ]:
def block_bootstrap_active_return(
        portfolio_returns,
        market_returns,
        block_months,
        repetitions,
        seed
):
    active_returns = (portfolio_returns - market_returns).dropna().to_numpy()
    generator = np.random.default_rng(seed)
    sample_size = len(active_returns)
    estimates = np.empty(repetitions)

    for repetition in range(repetitions):
        sample = []
        while len(sample) < sample_size:
            start = generator.integers(0, sample_size)
            positions = (start + np.arange(block_months)) % sample_size
            sample.extend(active_returns[positions])
        estimates[repetition] = np.mean(sample[:sample_size]) * 12

    return pd.Series({
        "annualized_active_return": active_returns.mean() * 12,
        "lower_95": np.quantile(estimates, 0.025),
        "upper_95": np.quantile(estimates, 0.975)
    })

In [ ]:
active_return_intervals = pd.DataFrame({
    method: block_bootstrap_active_return(
        walk_forward_returns[method],
        benchmark_returns,
        block_months=config["bootstrap_block_months"],
        repetitions=config["bootstrap_repetitions"],
        seed=config["random_seed"]
    )
    for method in methods
}).T

active_return_intervals

In [ ]:
performance_summary.to_csv(tables_directory / "walk_forward_performance.csv")
live_performance_summary.to_csv(tables_directory / "walk_forward_live_performance.csv")
active_return_intervals.to_csv(tables_directory / "active_return_intervals.csv")

## 4. Wealth and drawdowns

In [ ]:
starting_value = pd.DataFrame(
    1.0,
    index=[walk_forward_returns.index[0] - pd.offsets.MonthEnd(1)],
    columns=walk_forward_returns.columns
)
wealth = pd.concat([starting_value, (1 + walk_forward_returns).cumprod()])
drawdowns = wealth.div(wealth.cummax()).sub(1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
wealth.plot(ax=ax, logy=True)
ax.set_title("Historical walk-forward wealth")
ax.set_xlabel("Date")
ax.set_ylabel("Wealth Index (Logarithmic)")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(figures_directory / "walk_forward_wealth.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
drawdowns.plot(ax=ax)
ax.set_title("Historical walk-forward drawdowns")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(figures_directory / "walk_forward_drawdowns.png", dpi=150)
plt.show()

## Conclusion

The full walk-forward period and the common live period are reported separately. The block-bootstrap intervals show the uncertainty around average active returns and should be considered together with the point estimates.